# ANN Model Building

## Step 1: Data Loading, Feature Selection, and Stratified Splitting

In this step, we load the cleaned COVID-19 dataset, finalize the ANN input features, lock `DIED` as the target variable, and create reproducible train, validation, and test splits.

### Final Modeling Setup

- **Target variable:** `DIED`
- **Excluded feature:** `CLASIFFICATION_FINAL`
- **Input features:** `USMER`, `MEDICAL_UNIT`, `SEX`, `PATIENT_TYPE`, `PNEUMONIA`, `AGE`, `DIABETES`, `COPD`, `ASTHMA`, `INMSUPR`, `HIPERTENSION`, `OTHER_DISEASE`, `CARDIOVASCULAR`, `OBESITY`, `RENAL_CHRONIC`, `TOBACCO`
- **Split strategy:** Stratified `70%` training, `15%` validation, `15%` test

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().resolve().parent
DATA_PATH = ROOT_DIR / 'Data' / 'Cleaned_CovidData.csv'
ARTIFACT_DIR = ROOT_DIR / 'Model' / 'artifacts' / 'step1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    'USMER',
    'MEDICAL_UNIT',
    'SEX',
    'PATIENT_TYPE',
    'PNEUMONIA',
    'AGE',
    'DIABETES',
    'COPD',
    'ASTHMA',
    'INMSUPR',
    'HIPERTENSION',
    'OTHER_DISEASE',
    'CARDIOVASCULAR',
    'OBESITY',
    'RENAL_CHRONIC',
    'TOBACCO',
]
TARGET_COLUMN = 'DIED'
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(DATA_PATH)

required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing columns: {missing_columns}')

print('Dataset shape:', df.shape)
print('Total missing values:', int(df[required_columns].isnull().sum().sum()))
print('\nTarget distribution:')
print(df[TARGET_COLUMN].value_counts().sort_index())

In [ ]:
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

train_df = X_train.copy()
train_df[TARGET_COLUMN] = y_train.to_numpy()
validation_df = X_val.copy()
validation_df[TARGET_COLUMN] = y_val.to_numpy()
test_df = X_test.copy()
test_df[TARGET_COLUMN] = y_test.to_numpy()

In [ ]:
train_path = ARTIFACT_DIR / 'train.csv'
validation_path = ARTIFACT_DIR / 'validation.csv'
test_path = ARTIFACT_DIR / 'test.csv'

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

def split_summary(name, target):
    counts = target.value_counts().to_dict()
    total = int(target.shape[0])
    positive = int(counts.get(1, 0))
    negative = int(counts.get(0, 0))
    rate = positive / total if total else 0.0
    return {
        'split': name,
        'rows': total,
        'class_0_count': negative,
        'class_1_count': positive,
        'class_1_rate': round(rate, 6),
    }

summary_df = pd.DataFrame([
    split_summary('train', y_train),
    split_summary('validation', y_val),
    split_summary('test', y_test),
])

summary_df.to_csv(ARTIFACT_DIR / 'split_summary.csv', index=False)
summary_df

### Verified Step 1 Split Results

| Split | Rows | Survived (`0`) | Died (`1`) | Death Rate |
| --- | ---: | ---: | ---: | ---: |
| Train | 717,380 | 665,152 | 52,228 | 7.2804% |
| Validation | 153,724 | 142,532 | 11,192 | 7.2806% |
| Test | 153,725 | 142,533 | 11,192 | 7.2805% |

These values were verified from the implemented split artifacts and confirm that stratification preserved the class distribution across all subsets.

### Report Documentation for Step 1

In the first implementation step, the cleaned COVID-19 dataset was loaded and prepared for ANN modeling. The binary target variable `DIED` was selected as the prediction output, while `CLASIFFICATION_FINAL` was excluded from the feature set to reduce the risk of data leakage. A total of 16 input features were retained for modeling.

To support reliable model development, the dataset was divided into training, validation, and testing sets using a stratified split in the ratio of 70:15:15. Stratification was applied on the target variable so that the class distribution of patient survival and death remained consistent across all subsets. This step established a reproducible foundation for the preprocessing and model training stages that follow.